GSE275071: Single cell RNAseq analysis of PBMCs isolated from healthy adults subjeccted to sleep restriction

source: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE275071

In [ ]:
!pip install aeacus 

In [7]:
import os
import tarfile
import scanpy as sc
import anndata as ad
import aeacus
import numpy as np
import pandas as pd

In [ ]:
DATA_DIR = '/home/pandeyps/Prefix/aeacus/data/GSE275071'
os.makedirs(DATA_DIR, exist_ok=True)
os.chdir(DATA_DIR)

In [ ]:
RAW_URL = 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE275nnn/GSE275071/suppl/GSE275071_RAW.tar'
RAW_TAR = os.path.join(DATA_DIR, 'GSE275071_RAW.tar')
!curl -L -o {RAW_TAR} {RAW_URL}

In [ ]:
with tarfile.open(RAW_TAR, 'r') as tar:
    tar.extractall(path=DATA_DIR)

In [ ]:
h5_files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith('.h5')])

adatas = []

for h5_file in h5_files:
    h5_path = os.path.join(DATA_DIR, h5_file)
    gsm_id = h5_file.replace('.h5', '').replace('_filtered_feature_bc_matrix', '').replace('_Control_', '_').replace('_HS_', '_')
    sample_name = gsm_id
    condition = 'HS' if 'HS' in h5_file or 'habitual' in h5_file.lower() else ('SR' if 'SR' in h5_file or 'restriction' in h5_file.lower() else 'Unknown')
    if 'participant' in h5_file.lower() or 'GSM' in gsm_id:
        pass
    adata = sc.read_10x_h5(h5_path)
    adata.obs_names = [f"{gsm_id}_{b}_{i}" for i, b in enumerate(adata.obs_names)]
    if not adata.var_names.is_unique:
        adata.var_names_make_unique()
    adata.obs['sample'] = sample_name
    adata.obs['condition'] = condition
    adata.obs['gsm_id'] = gsm_id
    adatas.append(adata)
    print(f"  Cells: {adata.n_obs:,}, Genes: {adata.n_vars:,}")

In [10]:
adata = ad.concat(adatas, merge='same')
print(f"Combined dataset: {adata.n_obs:,} cells x {adata.n_vars:,} genes")

Combined dataset: 25,818 cells x 36,601 genes


In [ ]:
sc.pp.calculate_qc_metrics(adata, percent_top=None, log1p=False, inplace=True)
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

In [12]:
TEMP_H5AD = os.path.join(DATA_DIR, 'GSE275071_combined.h5ad')
adata.write_h5ad(TEMP_H5AD)
result = aeacus.Profiler(test_input=TEMP_H5AD, norm_type="cpm_log1p").load().profile()

print(f"Malignant cells: {(result.obs['malignancy_call'] == 'Malignant').sum()}")
print(f"Non-malignant cells: {(result.obs['malignancy_call'] == 'Normal').sum()}")

Model features: 3778
Missing features: 43 (1.14%)
Malignant cells: 15
Non-malignant cells: 25803
